In [4]:
import numpy as np
from tensorflow.keras.datasets import mnist
# MNIST contient des images de chiffres manuscrits.
# x_train : images d'entraînement
# y_train : étiquettes des images (0 à 9)
# x_test : images de test
# y_test : étiquettes de test
# Chaque image est de taille :
# 28 × 28 pixels
# Chargement du jeu de données MNIST
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Normalisation des images
# Les pixels d'une image sont compris entre :
#        0  → noir
#      255 → blanc
# Afin de faciliter l'apprentissage,
# on transforme ces valeurs entre 0 et 1.
x_train = x_train / 255.0
x_test = x_test / 255.0
# Pour simplifier le projet,
# nous n'utilisons qu'un petit nombre d'images.
x_train = x_train[:500]
y_train = y_train[:500]

x_test = x_test[:100]
y_test = y_test[:100]
# Initialisation du filtre de convolution
# Dans notre CNN, nous utiliserons :
#      un seul filtre 3 × 3
# Ce filtre représente les poids de la couche
# de convolution.
# Au début de l'apprentissage,
# ces poids sont choisis aléatoirement. Plus tard,ils seront modifiés grâce à
# la rétropropagation (Backpropagation).
filter_size = 3
conv_filter = np.random.randn(filter_size, filter_size) * 0.1

# Initialisation de la couche Dense
# Après :
# Convolution et Max Pooling
# notre image aura une taille de : 13 × 13
# Après Flatten : 13 × 13 = 169 neurones d'entrée.
# La couche de sortie possède :  10 neurones
# correspondant aux chiffres : 0 1 2 3 4 5 6 7 8 9

dense_weights = np.random.randn(169,10) * 0.1
dense_bias = np.zeros(10)

# Fonction : convolution()
# Cette fonction réalise la première étape du CNN :la convolution.
# Principe :
# Un filtre (appelé aussi kernel) de taille 3 × 3
# se déplace sur toute l'image.
# A chaque position :
# 1. On extrait une petite région 3 × 3 de l'image.
# 2. On multiplie chaque pixel par la valeur
#    correspondante du filtre.
# 3. On additionne tous les résultats.
# 4. La somme obtenue devient un pixel
#    de la Feature Map.
# Le filtre utilise les mêmes poids
# partout dans l'image.

def convolution(image, conv_filter):

    # Taille de l'image (28 × 28)
    image_height, image_width = image.shape

    # Taille du filtre (3 × 3)
    filter_height, filter_width = conv_filter.shape

    # Taille de la Feature Map obtenue
    output_height = image_height - filter_height + 1
    output_width = image_width - filter_width + 1

    # Création de la Feature Map remplie de zéros
    feature_map = np.zeros((output_height, output_width))

    # Le filtre se déplace ligne par ligne
    for i in range(output_height):

        # Puis colonne par colonne
        for j in range(output_width):

            # Extraction de la région 3 × 3
            region = image[i:i+filter_height,
                           j:j+filter_width]

            # Calcul de la convolution :
            # somme(region × filtre)
            feature_map[i, j] = np.sum(region * conv_filter)

    # Retourner la nouvelle image
    return feature_map

# Fonction : sigmoid()
# Après la convolution, les valeurs de la Feature Map
# peuvent être positives ou négatives.
# La fonction d'activation Sigmoid transforme chaque
# valeur en un nombre compris entre 0 et 1.

def sigmoid(x):

    return 1 / (1 + np.exp(-x))

    
# Fonction : max_pooling()
# Cette fonction réduit la taille de la Feature Map.
# Le Max Pooling parcourt l'image avec une fenêtre
# 2 × 2.
# Pour chaque fenêtre, on conserve uniquement la
# plus grande valeur.
# Cette opération permet :
# - de diminuer le nombre de calculs,
# - de réduire le nombre de paramètres,
# - de conserver les informations les plus importantes.

def max_pooling(feature_map, size=2):

    # Récupérer les dimensions de la Feature Map
    height, width = feature_map.shape
    # Calculer les dimensions de la nouvelle matrice
    output_height = height // size
    output_width = width // size
    # Créer la matrice qui contiendra le résultat
    pooled = np.zeros((output_height, output_width))

    # Parcourir la Feature Map avec une fenêtre 2 × 2
    for i in range(output_height):
        for j in range(output_width):
            # Extraire une région de taille 2 × 2
            region = feature_map[
                i*size:(i+1)*size,
                j*size:(j+1)*size
            ]

            # Conserver uniquement la valeur maximale
            pooled[i, j] = np.max(region)
    # Retourner la Feature Map réduite
    return pooled

# Fonction : flatten()
# Après les couches de convolution et de pooling,
# les données sont encore sous forme de matrice.
# La couche Dense attend un vecteur en entrée.
# Flatten transforme donc la matrice en une seule ligne de valeurs.
# Exemple : 13 × 13
# devient :169 valeurs

def flatten(feature_map):
    vector = []
    for row in feature_map:
        for value in row:
            vector.append(value)

    return np.array(vector)

# Fonction : dense()
# Cette fonction représente une couche Fully Connected.
# Après le Flatten, toutes les valeurs du vecteur sont
# connectées à chacun des neurones de sortie.
# Chaque neurone calcule : somme(x × poids) + biais
# Le résultat obtenu est appelé "score" (ou logit).
# Ces scores seront ensuite envoyés à la fonction
# Softmax afin d'obtenir des probabilités.

def dense(x, weights, bias):
    # Produit matriciel entre le vecteur d'entrée
    # et la matrice des poids
    output = np.dot(x, weights)
    # Ajout du biais
    output = output + bias
    return output
    

# Fonction : softmax()
# La couche Dense produit uniquement des scores.
# Ces scores peuvent être positifs ou négatifs
# et leur somme n'a aucune signification.
# Softmax transforme ces scores en probabilités.
# Les probabilités obtenues sont comprises entre
# 0 et 1, et leur somme est toujours égale à 1.
# La classe ayant la plus grande probabilité
# sera la prédiction du réseau.

def softmax(x):
    # Améliore la stabilité numérique
    x = x - np.max(x)

    # Calcul de l'exponentielle
    exp_x = np.exp(x)

    # Normalisation
    return exp_x / np.sum(exp_x)

# Fonction : forward()
# Cette fonction réalise toute la propagation avant
# (Forward Propagation) du CNN.
# A la fin, la fonction retourne les probabilités
# associées aux 10 chiffres (0 à 9).

def forward(image):

    # Couche de convolution
    conv = convolution(image, conv_filter)

    # Fonction d'activation
    activated = sigmoid(conv)

    # Max Pooling
    pooled = max_pooling(activated)

    # Transformation en vecteur
    vector = flatten(pooled)

    # Couche Dense
    scores = dense(vector, dense_weights, dense_bias)

    # Probabilités finales
    prediction = softmax(scores)

    return conv, activated, pooled, vector, scores, prediction

# Fonction : cross_entropy()
# Cette fonction calcule l'erreur entre
# la vraie classe et la prédiction du réseau.
def cross_entropy(y_true, y_pred):

    return -np.log(y_pred[y_true] + 1e-10)

# Fonction : backward()
# Cette fonction réalise la rétropropagation
# (Backpropagation).
# Son objectif est de corriger les poids de la
# couche Dense afin de réduire progressivement
# l'erreur du réseau.
# Les étapes sont :
#
# 1. Calculer l'erreur entre la prédiction
#    et la vraie classe.
#
# 2. Calculer les gradients des poids
#    et des biais.
#
# 3. Mettre à jour les paramètres grâce
#    au taux d'apprentissage (Learning Rate).

def backward(vector, prediction, label, learning_rate=0.01):

    global dense_weights, dense_bias

    # Copier les probabilités prédites
    error = prediction.copy()

    # Soustraire 1 à la probabilité de la vraie classe
    error[label] -= 1

    # Transformer le vecteur d'entrée en colonne
    vector = vector.reshape(-1, 1)

    # Calcul du gradient des poids
    gradient_weights = np.dot(vector, error.reshape(1, -1))

    # Calcul du gradient des biais
    gradient_bias = error

    # Mise à jour des poids
    dense_weights -= learning_rate * gradient_weights

    # Mise à jour des biais
    dense_bias -= learning_rate * gradient_bias


# Fonction : train()
# Cette fonction entraîne le réseau.
# Pour chaque image :
# 1. Propagation avant.
# 2. Calcul de la Loss.
# 3. Rétropropagation.
# 4. Mise à jour des poids.
def train(epochs=5):

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")
        
        for i in range(len(x_train)):

            image = x_train[i]
            label = y_train[i]
    
            # Propagation avant
            conv, activated, pooled, vector, scores, prediction = forward(image)
    
            # Calcul de la Loss
            loss = cross_entropy(label, prediction)
    
            # Mise à jour des paramètres
            backward(vector, prediction, label)
    
            # Afficher la Loss toutes les 50 images
            #if i % 50 == 0:
              #  print(f"Image {i} | Loss = {loss:.4f}")
        print(f"Loss finale : {loss:.4f}")

    
# Fonction : predict()
# Cette fonction réalise une prédiction sur une image.
# Le chiffre prédit correspond à la
# probabilité la plus élevée.
def predict(image):
    # Réaliser une propagation avant sur l'image
    # Seule la prédiction finale nous intéresse,
    _, _, _, _, _, prediction = forward(image)

    # Retourner l'indice de la plus grande probabilité.
    # Cet indice correspond au chiffre prédit.
    return np.argmax(prediction)

# Programme principal
print("===================================")
print("Début de l'entraînement du CNN")
print("===================================")

train()

print("\nEntraînement terminé.")

print("\n===== Test =====")

prediction = predict(x_test[3])

print("Prédiction :", prediction)

print("Vrai chiffre :", y_test[3]) 

Début de l'entraînement du CNN

Epoch 1/5
Loss finale : 2.8090

Epoch 2/5
Loss finale : 2.7959

Epoch 3/5
Loss finale : 2.7828

Epoch 4/5
Loss finale : 2.7698

Epoch 5/5
Loss finale : 2.7567

Entraînement terminé.

===== Test =====
Prédiction : 6
Vrai chiffre : 0
